In [1]:
from pathlib import Path
import torch
import open_clip

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMAGE_DIR = ROOT / "resultats_pdf" / "images"
TEXT_DIR  = ROOT / "resultats_pdf" / "textes"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
print("Appareil :", DEVICE)

c:\Users\bouho\Desktop\TP3_Rag\TP3_Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\bouho\Desktop\TP3_Rag\TP3_Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bouho\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, s

Appareil : cpu


In [3]:
from PIL import Image

@torch.inference_mode()
def encoder_textes(textes):
    tokens = tokenizer(textes).to(DEVICE)          # textes → identifiants de tokens [N, 77]
    v = model.encode_text(tokens).float()          # → vecteurs [N, 512]
    v = v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)  # normalisation L2
    return v.cpu()

@torch.inference_mode()
def encoder_images(chemins):
    tenseurs = []
    for c in chemins:
        with Image.open(c) as img:
            tenseurs.append(preprocess(img.convert("RGB")))
    lot = torch.stack(tenseurs).to(DEVICE)         # [N, 3, 224, 224]
    v = model.encode_image(lot).float()            # → [N, 512]
    v = v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return v.cpu()

v = encoder_textes(["the Transformer architecture"])
print(v.shape, "| norme =", v.norm().item())

torch.Size([1, 512]) | norme = 1.0


In [6]:
images = sorted(IMAGE_DIR.iterdir())
descriptions = [
    "the Transformer model architecture with encoder and decoder stacks",
    "scaled dot-product attention diagram",
    "multi-head attention with parallel attention layers",
    "a photo of a cat sitting on a sofa",
]

V_img = encoder_images(images)
V_txt = encoder_textes(descriptions)
similarites = V_img @ V_txt.T          # produit scalaire = cosinus

import pandas as pd
pd.DataFrame(similarites.numpy(),
             index=[p.name for p in images],
             columns=[d[:30] for d in descriptions]).round(3)

,the Transformer model architec,scaled dot-product attention d,multi-head attention with para,a photo of a cat sitting on a
page_003_image_128.png,0.294,0.246,0.276,0.048
page_004_image_182.png,0.273,0.317,0.260,0.113
page_004_image_183.png,0.274,0.455,0.330,0.112


In [7]:
from open_clip.tokenizer import SimpleTokenizer
bpe = SimpleTokenizer()

page = (TEXT_DIR / "page_003.txt").read_text(encoding="utf-8")
print("Tokens dans la page 3 :", len(bpe.encode(page)))   # bien plus que 77 !

def decouper_texte(texte, taille=60, chevauchement=10):
    tokens = bpe.encode(texte)
    pas = taille - chevauchement
    for debut in range(0, len(tokens), pas):
        fragment = bpe.decode(tokens[debut:debut + taille]).strip()
        if fragment:
            yield fragment
        if debut + taille >= len(tokens):
            break

chunks = list(decouper_texte(page))
print(len(chunks), "chunks\n---\n", chunks[0], "\n---\n", chunks[1])

Tokens dans la page 3 : 415
9 chunks
---
 figure 1 : the transformer - model architecture . the transformer follows this overall architecture using stacked self - attention and point - wise , fully connected layers for both the encoder and decoder , shown in the left and right halves of figure 1 , respectively . 3 . 1 encoder and decoder stacks 
---
 . 3 . 1 encoder and decoder stacks encoder : the encoder is composed of a stack of n = 6 identical layers . each layer has two sub - layers . the first is a multi - head self - attention mechanism , and the second is a simple , position - wise fully connected
